# 01 Supplement - CanToDrawer Videos

Notebook này bổ sung video cho subset `gr1_arms_waist.CanToDrawer` sau khi đã tải `data/` + `meta/`.

Mặc định notebook **không tải full video** để tránh tràn disk Kaggle. Nó chỉ tải một camera view và một số chunk đầu để dùng cho phần minh họa / VLM feature extraction đại diện.

Nếu muốn tải nhiều hơn, tăng `CHUNK_IDS`, nhưng phải kiểm tra disk trước.


In [1]:
!pip install -q huggingface_hub


In [2]:
from huggingface_hub import login, whoami
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token)
print(whoami()["name"])


thkt2110


In [3]:
from pathlib import Path
import json
import os
import shutil

repo_id = "nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim"
subset_name = "gr1_arms_waist.CanToDrawer"

# Camera an toàn để bắt đầu. Không dùng videos/** vì sẽ tải quá nhiều.
camera_key = "observation.images.ego_view"

# Smoke-test an toàn: chỉ tải vài chunk đầu. Tăng dần nếu còn disk.
# Đổi thành None nếu thật sự muốn thử tải toàn bộ camera view, nhưng có rủi ro tràn disk.
CHUNK_IDS = [0, 1, 2]

local_dir = Path("/kaggle/working/gr00t_x_embodiment_sim_video_supplement")
report_path = Path("/kaggle/working/video_supplement_report_CanToDrawer.json")

# Bật True nếu muốn xóa phần video supplement lỗi trước đó trong cùng session.
CLEAN_EXISTING_SUPPLEMENT = False

def folder_size_gb(path):
    path = Path(path)
    if not path.exists():
        return 0.0
    total = sum(p.stat().st_size for p in path.rglob("*") if p.is_file())
    return total / (1024 ** 3)

if CLEAN_EXISTING_SUPPLEMENT and local_dir.exists():
    shutil.rmtree(local_dir)

local_dir.mkdir(parents=True, exist_ok=True)

print("Subset:", subset_name)
print("Camera:", camera_key)
print("Chunk IDs:", CHUNK_IDS)
print("Output:", local_dir)
os.system("df -h /kaggle/working")


Subset: gr1_arms_waist.CanToDrawer
Camera: observation.images.ego_view
Chunk IDs: [0, 1, 2]
Output: /kaggle/working/gr00t_x_embodiment_sim_video_supplement
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  120K   20G   1% /kaggle/working


0

In [4]:
from huggingface_hub import snapshot_download

if CHUNK_IDS is None:
    allow_patterns = [f"{subset_name}/videos/**/{camera_key}/**"]
else:
    allow_patterns = [
        f"{subset_name}/videos/chunk-{chunk_id:03d}/{camera_key}/**"
        for chunk_id in CHUNK_IDS
    ]

print("Allow patterns:")
for p in allow_patterns:
    print("-", p)

snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    allow_patterns=allow_patterns,
    local_dir=str(local_dir),
    max_workers=2,
)

print("Download finished.")
os.system("df -h /kaggle/working")
print("Supplement size GB:", round(folder_size_gb(local_dir), 3))


Allow patterns:
- gr1_arms_waist.CanToDrawer/videos/chunk-000/observation.images.ego_view/**
- gr1_arms_waist.CanToDrawer/videos/chunk-001/observation.images.ego_view/**
- gr1_arms_waist.CanToDrawer/videos/chunk-002/observation.images.ego_view/**


Fetching ... files: 0it [00:00, ?it/s]

Download finished.
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  6.3G   14G  32% /kaggle/working
Supplement size GB: 6.198


In [5]:
subset_path = local_dir / subset_name
videos_dir = subset_path / "videos"
video_files = sorted(videos_dir.rglob("*.mp4")) if videos_dir.exists() else []

report = {
    "repo_id": repo_id,
    "subset_name": subset_name,
    "camera_key": camera_key,
    "chunk_ids": CHUNK_IDS,
    "local_dir": str(local_dir),
    "subset_path": str(subset_path),
    "exists": subset_path.exists(),
    "size_gb": round(folder_size_gb(subset_path), 3),
    "num_video_files": len(video_files),
    "first_video_files": [str(p.relative_to(subset_path)) for p in video_files[:10]],
}

with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))
print("Saved report to:", report_path)


{
  "repo_id": "nvidia/PhysicalAI-Robotics-GR00T-X-Embodiment-Sim",
  "subset_name": "gr1_arms_waist.CanToDrawer",
  "camera_key": "observation.images.ego_view",
  "chunk_ids": [
    0,
    1,
    2
  ],
  "local_dir": "/kaggle/working/gr00t_x_embodiment_sim_video_supplement",
  "subset_path": "/kaggle/working/gr00t_x_embodiment_sim_video_supplement/gr1_arms_waist.CanToDrawer",
  "exists": true,
  "size_gb": 6.198,
  "num_video_files": 3000,
  "first_video_files": [
    "videos/chunk-000/observation.images.ego_view/episode_000000.mp4",
    "videos/chunk-000/observation.images.ego_view/episode_000001.mp4",
    "videos/chunk-000/observation.images.ego_view/episode_000002.mp4",
    "videos/chunk-000/observation.images.ego_view/episode_000003.mp4",
    "videos/chunk-000/observation.images.ego_view/episode_000004.mp4",
    "videos/chunk-000/observation.images.ego_view/episode_000005.mp4",
    "videos/chunk-000/observation.images.ego_view/episode_000006.mp4",
    "videos/chunk-000/observatio

## Cách dùng output này

Output của notebook này chỉ chứa video supplement:

```text
/kaggle/working/gr00t_x_embodiment_sim_video_supplement/
```

Khi cần dùng cho notebook VLM/visual feature sau này, attach cả hai Kaggle Dataset:

1. Dataset chứa `data/` + `meta/` của `CanToDrawer`.
2. Dataset video supplement từ notebook này.

Không nên gộp full video vào cùng notebook download data/meta nếu disk Kaggle chỉ khoảng 20GB output.
